# Notebook 01: Dataset exploration

**Project:** Lightweight machine-learning-based cybersecurity threat detection for resource-constrained hardware
**Author:** Anush Jindal

A first look at the CICIDS2017 Wednesday file before any cleaning or modelling: size,
column names and types, missing and infinite values, duplicates, and the class
distribution. Nothing is changed here except the column names; the cleaning itself is
done in notebook 02 based on what is found below.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Paths: the repo layout when run locally, the Drive folder when run in Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/threat detector")
except ImportError:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
RAW_CSV = DATA_DIR / "Wednesday-workingHours.pcap_ISCX.csv"

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Raw CSV:", RAW_CSV)

## 2. Load the dataset

`Wednesday-workingHours.pcap_ISCX.csv` from CICIDS2017 (see `data/README.md`). It holds one
working day of traffic: normal traffic plus four DoS attacks and a short Heartbleed attack.

In [ ]:
df = pd.read_csv(RAW_CSV)
print("Loaded:", type(df).__name__)

## 3. Shape

In [ ]:
df.shape

692,703 rows (one per network flow) and 79 columns: 78 flow features plus the label.

## 4. First rows

In [ ]:
df.head()

## 5. Column names

The raw file has a leading space in most column names (for example `" Label"`), so
`df["Label"]` fails with a `KeyError`. Printing the names with `repr()` makes the spaces
visible; they are then stripped. This only renames columns and does not change any data.

In [ ]:
print(f"Number of columns: {len(df.columns)}\n")
for col in df.columns:
    print(repr(col))  # repr() shows quotes, so leading/trailing spaces are visible

In [ ]:
df.columns = df.columns.str.strip()
print("Column names stripped.")

## 6. Data types

All 78 features are numeric (`int64` or `float64`); only `Label` is text.

In [ ]:
df.info()

## 7. Summary statistics

The rate columns (`Flow Bytes/s`, `Flow Packets/s`) reach `inf`: a flow with a duration
of 0 gives a division by zero. Several columns are zero in every row; those are dealt with
during feature selection in notebook 03.

In [ ]:
df.describe()

## 8. Missing and infinite values

In [ ]:
missing = df.isnull().sum()
missing_only = missing[missing > 0].sort_values(ascending=False)
print(f"Columns with missing values: {len(missing_only)} out of {len(df.columns)}\n")
missing_only

Only `Flow Bytes/s` has missing values (1,008 rows). `isnull()` does not detect `inf`,
so infinite values are counted separately.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
inf_counts = np.isinf(numeric_df).sum()
inf_only = inf_counts[inf_counts > 0].sort_values(ascending=False)
print(f"Numeric columns containing infinite values: {len(inf_only)}\n")
inf_only

Two columns contain infinite values: `Flow Packets/s` (1,297 rows) and `Flow Bytes/s`
(289 rows). Since 1,008 + 289 = 1,297, the missing `Flow Bytes/s` values are most likely
the same zero-duration flows as the infinite `Flow Packets/s` ones; this fits notebook 02,
where no further rows are dropped for missing values once the infinite rows are gone.

## 9. Duplicate rows

In [ ]:
num_duplicates = df.duplicated().sum()
print(f"Number of fully duplicated rows: {num_duplicates}")
print(f"That's {num_duplicates / len(df) * 100:.2f}% of all rows")

81,909 rows (11.82%) are exact duplicates of another row.

## 10. Class distribution

In [ ]:
LABEL_COLUMN = "Label"

assert LABEL_COLUMN in df.columns, f"'{LABEL_COLUMN}' not found. Actual columns: {list(df.columns)}"

print("Raw counts:")
print(df[LABEL_COLUMN].value_counts())
print("\nAs percentages:")
print((df[LABEL_COLUMN].value_counts(normalize=True) * 100).round(2))

In [ ]:
plt.figure(figsize=(10, 5))
df[LABEL_COLUMN].value_counts().plot(kind="bar")
plt.title("Class distribution")
plt.ylabel("Number of flows")
plt.xlabel("Label")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

| Label | Flows | Share |
|---|---|---|
| BENIGN | 440,031 | 63.52% |
| DoS Hulk | 231,073 | 33.36% |
| DoS GoldenEye | 10,293 | 1.49% |
| DoS slowloris | 5,796 | 0.84% |
| DoS Slowhttptest | 5,499 | 0.79% |
| Heartbleed | 11 | 0.0016% |

BENIGN and DoS Hulk make up about 97% of the rows. The other three DoS attacks are rare,
and Heartbleed (11 rows) is too rare to score reliably. Because of this imbalance,
accuracy alone is not a useful measure: per-class recall and macro F1 are reported in
the later notebooks. The classes are kept separate (not merged into "attack"), so each
attack's detection can be tracked as the model shrinks. What each attack is:
`research/attack_categories.md`.

## Summary

| Check | Result |
|---|---|
| File | `Wednesday-workingHours.pcap_ISCX.csv` |
| Size | 692,703 rows x 79 columns (78 features + `Label`) |
| Column names | Leading spaces in the raw file, stripped |
| Missing values | `Flow Bytes/s` only, 1,008 rows |
| Infinite values | `Flow Packets/s` 1,297 rows, `Flow Bytes/s` 289 rows |
| Duplicates | 81,909 rows (11.82%) |
| Classes | BENIGN plus 5 attacks; Heartbleed has only 11 rows |

These findings set the cleaning steps in `02_preprocessing.ipynb`.